# Halo catalogue to HDF5 conversion

Notebook for converting halo catalogues into HDF5 files with a unified format, optionally extracting only selected properties of interest to simplify subsequent analysis.

In [1]:
from os.path import join, exists

import numpy as np
import matplotlib.pyplot as plt
from h5py import File
from tqdm import tqdm

import hdf5plugin
import csiborgtools

%load_ext autoreload
%autoreload 2

paths = csiborgtools.read.Paths(**csiborgtools.paths_glamdring)
# paths = csiborgtools.read.Paths(**csiborgtools.paths_rusty)

## CSiBORG1 

In [2]:
src_dir = "/mnt/extraspace/rstiskalek/csiborg_postprocessing/SOcat"
stem = "csiborg1"
out_path = "csiborg1_fof.hdf5"

with File(out_path, "w") as fout:
    for nsim in tqdm(paths.get_ics("csiborg1")):
        src = join(src_dir, f"{stem}_{nsim}.hdf5")
        if not exists(src):
            print(f"[skip] missing: {src}")
            continue

        with File(src, "r") as fin:
            grp = fout.create_group(str(nsim))

            # copy all top-level datasets
            for name in fin.keys():
                grp.create_dataset(
                    name, data=fin[name][...],
                    compression="gzip", shuffle=True
                )

            # copy file-level attributes
            for key, val in fin.attrs.items():
                grp.attrs[key] = val

print(f"Unified catalogue written to {out_path}")

100%|██████████| 101/101 [00:29<00:00,  3.43it/s]

Unified catalogue written to csiborg1_fof.hdf5


In [3]:
!pwd

/mnt/users/rstiskalek/csiborgtools/notebooks


## CSiBORG2

In [9]:
from tqdm import tqdm

with File("csiborg2_fof.hdf5", 'w') as f:
    for nsim in tqdm(paths.get_ics("csiborg2_main")):
        cat = csiborgtools.read.CSiBORG2Catalogue(nsim, 99, "main", paths=paths)
        grp = f.create_group(str(nsim))

        grp["Coordinates"] = cat['cartesian_pos']
        grp["Velocities"] = cat['cartesian_vel']
        grp["GroupMass"] = cat.totmass
        grp["Group_M_Crit200"] = cat["Group_M_Crit200"] * 1e10
        grp["Group_R_Crit200"] = cat["Group_R_Crit200"]
        grp["Group_M_Crit500"] = cat["Group_M_Crit500"] * 1e10
        grp["Group_R_Crit500"] = cat["Group_R_Crit500"]


  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 20/20 [00:15<00:00,  1.26it/s]


### Manticore 

In [7]:
reader = csiborgtools.read.CSiBORG3Catalogue(0, 130, paths, )

In [8]:
reader.keys()

['cartesian_pos',
 'spherical_pos',
 'galactic_pos',
 'dist',
 'cartesian_redshiftspace_pos',
 'spherical_redshiftspace_pos',
 'redshiftspace_dist',
 'cartesian_vel',
 'particle_offsetnpart',
 'totmass',
 'index',
 'lagpatch_coordinates',
 'lagpatch_radius',
 'GroupFirstSub',
 'GroupContamination',
 'GroupNsubs',
 'Group_M_Crit200',
 'Group_M_Crit500',
 'Group_R_Crit200',
 'Group_R_Crit500']

In [16]:
from tqdm import tqdm, trange

with File("manticore_fof.hdf5", 'w') as f:
    for nsim in trange(50):
        cat = csiborgtools.read.CSiBORG3Catalogue(
            nsim, 130, paths=paths, verbose=False)
        grp = f.create_group(str(nsim))

        mask = cat.totmass > 1e12

        grp["Coordinates"] = cat['cartesian_pos'][mask]
        grp["Velocities"] = cat['cartesian_vel'][mask]
        grp["GroupMass"] = cat.totmass[mask]
        grp["Group_M_Crit200"] = cat["Group_M_Crit200"][mask] * 1e10
        grp["Group_R_Crit200"] = cat["Group_R_Crit200"][mask]
        grp["Group_M_Crit500"] = cat["Group_M_Crit500"][mask] * 1e10
        grp["Group_R_Crit500"] = cat["Group_R_Crit500"][mask]

  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [01:15<00:00,  1.52s/it]


In [17]:
!du -h manticore_fof.hdf5

136M	manticore_fof.hdf5
